<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok

In [7]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio
import uvicorn

nest_asyncio.apply()

app = FastAPI(
    title="Day 23 AI RAG API",
    description="API for connecting the RAG system to a frontend",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QuestionRequest(BaseModel):
    question: str

def run_rag(question: str):
    answer = f"I received your question: {question}"

    sources = [
        {
            "title": "Temporary Source",
            "content": "This source will be replaced with your actual RAG sources."
        }
    ]

    return {
        "answer": answer,
        "sources": sources
    }

@app.post("/ask")
async def ask_question(request: QuestionRequest):
    if not request.question.strip():
        return {
            "answer": "Please enter a question.",
            "sources": []
        }

    result = run_rag(request.question)
    return result

@app.get("/")
async def root():
    return {
        "message": "Day 23 AI RAG API is running!"
    }

print("FastAPI application created successfully!")
print("Available endpoints:")
print("GET  /")
print("POST /ask")
print("GET  /docs")

FastAPI application created successfully!
Available endpoints:
GET  /
POST /ask
GET  /docs


In [8]:
import threading
import time
import uvicorn

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

time.sleep(3)

print("FastAPI server started on port 8000")

INFO:     Started server process [695]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


FastAPI server started on port 8000


In [9]:
import requests

url = "http://127.0.0.1:8000/ask"

payload = {
    "question": "What is artificial intelligence?"
}

response = requests.post(url, json=payload)

print("Status code:", response.status_code)
print("Response:", response.json())

INFO:     127.0.0.1:35798 - "POST /ask HTTP/1.1" 200 OK
Status code: 200
Response: {'answer': 'I received your question: What is artificial intelligence?', 'sources': [{'title': 'Temporary Source', 'content': 'This source will be replaced with your actual RAG sources.'}]}


In [11]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [12]:
import subprocess
import time

cloudflared_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("Cloudflare Tunnel started.")

Cloudflare Tunnel started.


In [13]:
import re

for _ in range(30):
    line = cloudflared_process.stdout.readline()

    if "trycloudflare.com" in line:
        print(line.strip())
        break

2026-09-05T14:28:11Z INF Requesting new quick Tunnel on trycloudflare.com...


In [15]:
import subprocess
import time

cloudflared_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

for _ in range(30):
    line = cloudflared_process.stdout.readline()
    if "trycloudflare.com" in line:
        print(line.strip())
        break

2026-09-05T14:29:14Z INF Requesting new quick Tunnel on trycloudflare.com...


In [17]:
import subprocess
import time
import re

cloudflared_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

public_url = None

for _ in range(40):
    line = cloudflared_process.stdout.readline()

    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)

    if match:
        public_url = match.group(0)
        break

print("Public API URL:", public_url)

Public API URL: https://news-bracket-peter-mobiles.trycloudflare.com


In [22]:
import requests

PUBLIC_API_URL = "https://news-bracket-peter-mobiles.trycloudflare.com"

response = requests.get(f"{PUBLIC_API_URL}/")

print("Status code:", response.status_code)
print("Response:", response.json())

INFO:     35.204.99.21:0 - "GET / HTTP/1.1" 200 OK
Status code: 200
Response: {'message': 'Day 23 AI RAG API is running!'}


In [23]:
from fastapi.responses import StreamingResponse
import asyncio

async def generate_response(question: str):
    result = run_rag(question)
    answer = result["answer"]

    for word in answer.split():
        yield word + " "
        await asyncio.sleep(0.08)

@app.post("/ask-stream")
async def ask_stream(request: QuestionRequest):
    if not request.question.strip():
        return StreamingResponse(
            iter(["Please enter a question."]),
            media_type="text/plain"
        )

    return StreamingResponse(
        generate_response(request.question),
        media_type="text/plain"
    )

print("Streaming endpoint created: POST /ask-stream")

Streaming endpoint created: POST /ask-stream


In [24]:
import os
import signal

os.kill(os.getpid(), signal.SIGINT)

KeyboardInterrupt: 

In [25]:
import threading
import time
import uvicorn

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

time.sleep(3)

print("FastAPI server restarted successfully.")
print("Streaming endpoint: /ask-stream")

INFO:     Started server process [695]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


FastAPI server restarted successfully.
Streaming endpoint: /ask-stream


In [26]:
import requests

response = requests.post(
    f"{PUBLIC_API_URL}/ask-stream",
    json={"question": "What is artificial intelligence?"}
)

print("Status code:", response.status_code)
print("Response:", response.text)

INFO:     35.204.99.21:0 - "POST /ask-stream HTTP/1.1" 200 OK
Status code: 200
Response: I received your question: What is artificial intelligence? 


In [27]:
answer = """# Artificial Intelligence

Artificial Intelligence is **technology that enables computers to perform tasks that normally require human intelligence**.

### Examples

- Chatbots
- Recommendation systems
- Image recognition

```python
print("AI")
```"""

In [28]:
from fastapi.responses import StreamingResponse
import asyncio
import json

async def generate_response(question: str):
    result = run_rag(question)

    answer = result.get("answer", "")
    sources = result.get("sources", [])

    for word in answer.split():
        yield json.dumps({
            "type": "token",
            "content": word + " "
        }) + "\n"

        await asyncio.sleep(0.08)

    yield json.dumps({
        "type": "sources",
        "sources": sources
    }) + "\n"

@app.post("/ask-stream")
async def ask_stream(request: QuestionRequest):
    if not request.question.strip():
        return StreamingResponse(
            iter([
                json.dumps({
                    "type": "error",
                    "message": "Please enter a question."
                }) + "\n"
            ]),
            media_type="application/x-ndjson"
        )

    return StreamingResponse(
        generate_response(request.question),
        media_type="application/x-ndjson"
    )

print("Streaming endpoint updated successfully.")

Streaming endpoint updated successfully.
